In [4]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import pickle
import os

In [6]:
df = pd.read_csv("../data/raw/student_lifestyle_dataset.csv")
df = df.drop(columns=["Student_ID"])

print(f"Shape: {df.shape}")
df.head()

Shape: (2000, 7)


,Study_Hours_Per_Day,Extracurricular_Hours_Per_Day,Sleep_Hours_Per_Day,Social_Hours_Per_Day,Physical_Activity_Hours_Per_Day,GPA,Stress_Level
0,6.9,3.8,8.7,2.8,1.8,2.99,Moderate
1,5.3,3.5,8.0,4.2,3.0,2.75,Low
2,5.1,3.9,9.2,1.2,4.6,2.67,Low
3,6.5,2.1,7.2,1.7,6.5,2.88,Moderate
4,8.1,0.6,6.5,2.2,6.6,3.51,High


In [7]:
stress_mapping = {'Low': 0, 'Moderate': 1, 'High': 2}
df['Stress_Level_Encoded'] = df['Stress_Level'].map(stress_mapping)

print("Encoding mapping:", stress_mapping)
print(df[['Stress_Level', 'Stress_Level_Encoded']].value_counts().sort_index())

Encoding mapping: {'Low': 0, 'Moderate': 1, 'High': 2}
Stress_Level  Stress_Level_Encoded
High          2                       1029
Low           0                        297
Moderate      1                        674
Name: count, dtype: int64


In [9]:
features = ['Study_Hours_Per_Day', 'Extracurricular_Hours_Per_Day',
            'Sleep_Hours_Per_Day', 'Social_Hours_Per_Day',
            'Physical_Activity_Hours_Per_Day', 'GPA']

X = df[features]
y = df['Stress_Level_Encoded']

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"\nClass distribution:\n{y.value_counts().sort_index()}")

X shape: (2000, 6)
y shape: (2000,)

Class distribution:
Stress_Level_Encoded
0     297
1     674
2    1029
Name: count, dtype: int64


In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y          
)

print(f"X_train : {X_train.shape}")
print(f"X_test  : {X_test.shape}")
print(f"\nTrain class distribution:\n{y_train.value_counts().sort_index()}")
print(f"\nTest class distribution:\n{y_test.value_counts().sort_index()}")

X_train : (1600, 6)
X_test  : (400, 6)

Train class distribution:
Stress_Level_Encoded
0    238
1    539
2    823
Name: count, dtype: int64

Test class distribution:
Stress_Level_Encoded
0     59
1    135
2    206
Name: count, dtype: int64


In [11]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)   
X_test_scaled  = scaler.transform(X_test)        

X_train_scaled = pd.DataFrame(X_train_scaled, columns=features, index=X_train.index)
X_test_scaled  = pd.DataFrame(X_test_scaled,  columns=features, index=X_test.index)

print(f"\nX_train_scaled mean :\n{X_train_scaled.mean().round(3)}")
print(f"\nX_train_scaled std  :\n{X_train_scaled.std().round(3)}")


X_train_scaled mean :
Study_Hours_Per_Day                0.0
Extracurricular_Hours_Per_Day      0.0
Sleep_Hours_Per_Day                0.0
Social_Hours_Per_Day              -0.0
Physical_Activity_Hours_Per_Day   -0.0
GPA                                0.0
dtype: float64

X_train_scaled std  :
Study_Hours_Per_Day                1.0
Extracurricular_Hours_Per_Day      1.0
Sleep_Hours_Per_Day                1.0
Social_Hours_Per_Day               1.0
Physical_Activity_Hours_Per_Day    1.0
GPA                                1.0
dtype: float64


In [12]:
os.makedirs('../data/processed', exist_ok=True)

# Save splits
X_train_scaled.to_csv('../data/processed/X_train.csv', index=False)
X_test_scaled.to_csv('../data/processed/X_test.csv',   index=False)
y_train.to_csv('../data/processed/y_train.csv',         index=False)
y_test.to_csv('../data/processed/y_test.csv',           index=False)

# Save cleaned full dataset (encoded)
df.to_csv('../data/processed/df_cleaned.csv', index=False)

# Save scaler 
os.makedirs('../src', exist_ok=True)
with open('../src/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)